In [10]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import geopandas as gpd

In [14]:
# setting file paths 

dir = "/global/scratch/users/yougsanghvi"

# setting suicide panel file path
suicide_proj_folder = "data"
suicide_panel_folderp = os.path.join("merged", "USA")
suicide_panel_filen = "USA_adm2_1968_2004_monthly.dta"
suicide_panel_filep = os.path.join(dir, suicide_proj_folder, suicide_panel_folderp, suicide_panel_filen)

# Define paths for usa county shapefile
usa_county_dir = os.path.join(dir, "shapefiles")
usa_county_filename = "tl_2016_us_county_mortality.shp"
usa_county_path = os.path.join(usa_county_dir, usa_county_filename)

# reading in required files  
suicide_panel_data = pd.read_stata(suicide_panel_filep)
usa_counties = gpd.read_file(usa_county_path)

In [ ]:
# Filter and select columns
filtered_suicide = suicide_panel_data[
    (suicide_panel_data["agegroup"] == 0) & (suicide_panel_data["gender"] == 0)
][[
    'fipsst', 'fipscty', 'fips', 'year', 'month', 'period', 'num_of_suicide',
    'suiciderate', 'population', 'nsuicide_b1988', 'nsuicide_a1989',
    'sol_rate_adj', 'sol_n_suicide', 'pop', 'adm1_id', 'adm2_id',
    'statename', 'countyname', 'tavg_poly1_aw', 'tavg_poly2_aw',
    'tavg_poly3_aw', 'tavg_poly4_aw', 'tavg_poly5_aw'
]]


    fipsst  fipscty    fips    year  month  period  num_of_suicide  \
3      1.0      1.0  1001.0  1968.0      1     1.0               0   
11     1.0      1.0  1001.0  1968.0      2     2.0               0   
18     1.0      1.0  1001.0  1968.0      3     3.0               0   
33     1.0      1.0  1001.0  1968.0      4     4.0               0   
36     1.0      1.0  1001.0  1968.0      5     5.0               1   

    suiciderate  population  nsuicide_b1988  ...      pop  adm1_id  adm2_id  \
3      0.000000     23214.0        2.380952  ...  23315.0      1.0   1001.0   
11     0.000000     23214.0        2.380952  ...  23315.0      1.0   1001.0   
18     0.000000     23214.0        2.380952  ...  23315.0      1.0   1001.0   
33     0.000000     23214.0        2.380952  ...  23315.0      1.0   1001.0   
36     4.307745     23214.0        2.380952  ...  23315.0      1.0   1001.0   

    statename  countyname  tavg_poly1_aw tavg_poly2_aw tavg_poly3_aw  \
3     Alabama     Autauga       

In [ ]:
# Step 1: Create a continuous date variable — number of months since a reference point
filtered_suicide['date_num'] = (filtered_suicide['year'] - filtered_suicide['year'].min()) * 12 + filtered_suicide['month']

# Step 2 & 3: Define a function to regress suiciderate on date_num and get slope
def get_slope(df):
    # Drop NA in suiciderate or date_num for regression
    df = df.dropna(subset=['suiciderate', 'date_num'])
    if len(df) < 2:
        return np.nan  # not enough data for regression
    X = sm.add_constant(df['date_num'])
    y = df['suiciderate']
    model = sm.OLS(y, X).fit()
    return model.params['date_num']  # slope coefficient

# Step 4: Group by county (using 'fips') and compute slope
slopes = filtered_suicide.groupby('fips').apply(get_slope).reset_index()
slopes.columns = ['fips', 'deltasuicide']

# 51 counties don't have any data on suiciderates
# Dropping these counties for now, but must be checked later 


deltasuicide    52
fips             0
dtype: int64


/tmp/ipykernel_3857876/1043454169.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  slopes = filtered_suicide.groupby('fips').apply(get_slope).reset_index()


In [25]:
# Get fips codes with NaN slope
nan_fips = slopes[slopes['deltasuicide'].isna()]['fips']

# Filter original data for those counties
nan_counties_data = filtered_suicide[filtered_suicide['fips'].isin(nan_fips)]

# Group by fips and calculate stats
nan_summary = nan_counties_data.groupby('fips').agg(
    avg_suiciderate=('suiciderate', 'mean'),
    avg_num_suicides=('num_of_suicide', 'mean'),
    num_missing_suiciderate=('suiciderate', lambda x: x.isna().sum()),
    num_nonmissing_suiciderate=('suiciderate', lambda x: x.notna().sum())
).reset_index()

print(nan_summary)



       fips  avg_suiciderate  avg_num_suicides  num_missing_suiciderate  \
0    2001.0              NaN          0.408730                      252   
1    2003.0              NaN          0.515873                      252   
2    2005.0              NaN          0.607143                      252   
3    2007.0              NaN          0.396825                      252   
4    2009.0              NaN          0.107143                      252   
5    2010.0              NaN          0.075397                      252   
6    2011.0              NaN          0.079365                      252   
7    2013.0              NaN         10.960317                      252   
8    2015.0              NaN          0.567460                      252   
9    2017.0              NaN          0.408730                      252   
10   2019.0              NaN          4.158730                      252   
11   2021.0              NaN          0.698413                      252   
12   2023.0              

In [ ]:
# Convert slopes['fips'] to string and zero-pad to 5 digits (typical GEOID format)
slopes['fips_str'] = slopes['fips'].astype(int).astype(str).str.zfill(5)

# Merge on string keys
usa_counties_with_slopes = usa_counties.merge(
    slopes, 
    left_on='GEOID', 
    right_on='fips_str', 
    how='left'
)

matched_rows = usa_counties_with_slopes['fips'].notna().sum()
total_rows = len(usa_counties_with_slopes)
print(f"Matched {matched_rows} out of {total_rows} shapefile counties ({100 * matched_rows / total_rows:.2f}%)")

# Matched 3122 out of 3228 shapefile counties (96.72%)
# For now, ignoring the unmatched columns but must be checked!!


0    01001
1    01003
2    01005
3    01007
4    01009
Name: fips_str, dtype: object
0    01001
1    01003
2    01005
3    01007
4    01009
Name: GEOID, dtype: object
Matched 3122 out of 3228 shapefile counties (96.72%)


In [20]:
na_counts = usa_counties_with_slopes.isna().sum()
print(na_counts.sort_values(ascending=False))


METDIVFP        3115
CSAFP           1997
CBSAFP          1329
deltasuicide     139
fips_str         106
fips             106
NAMELSAD           0
NAME               0
GEOID              0
COUNTYNS           0
ID_2               0
ID_1               0
MTFCC              0
LSAD               0
ALAND              0
FUNCSTAT           0
CLASSFP            0
AWATER             0
ID_0               0
INTPTLON           0
INTPTLAT           0
geometry           0
dtype: int64
